# CSE304 X-ray — Colab 실행 노트북 v3.1 (폴더 경로 오류 수정)

Colab 실행을 위한 노트북입니다. 분석 코드는 ZIP의 `src/xraydm/*.py`에 있습니다.

- Drive `MyDrive/CSE304_Xray`에 `Code_v3.zip` **또는** `CSE304_All_Files_v3.zip` 업로드
- `input_zips` 안에 KAMP 원본 8개 + `xray-labels-2026-11-10.zip` 1개
- 첫 실행은 `FAST_TEST=True, USE_DEEP=False`로 테스트


In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, io, os, shutil

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/CSE304_Xray')
INPUT = BASE / 'input_zips'
OUTPUT = BASE / 'results' / 'domain_v3'
ROOT = Path('/content')
WORK = ROOT / 'CSE304_Xray_Colab_Repo'

# 지원 파일명: 전체 ZIP, Code_v3.zip, 기존 정식 레포지토리 ZIP.
choices = [
    BASE / 'CSE304_Xray_DomainMining_Colab_Repo_v3.zip',
    BASE / 'Code_v3.zip',
    BASE / 'CSE304_All_Files_v3.zip',
    BASE / 'CSE304_Downloads' / 'Code_v3.zip',
]
REPO_ZIP = next((p for p in choices if p.is_file()), None)
if REPO_ZIP is None:
    raise FileNotFoundError(
        f'코드 ZIP을 찾을 수 없습니다. {BASE}/Code_v3.zip 또는 '
        f'{BASE}/CSE304_All_Files_v3.zip을 업로드하세요.\n'
        f'폴더 목록: {[x.name for x in BASE.iterdir()] if BASE.exists() else "BASE 폴더 없음"}'
    )

# 통합 ZIP이면 안쪽 Code_v3.zip만 꺼내서 사용.
if REPO_ZIP.name == 'CSE304_All_Files_v3.zip':
    with zipfile.ZipFile(REPO_ZIP) as outer:
        matches=[n for n in outer.namelist() if Path(n).name=='Code_v3.zip']
        if len(matches)!=1:
            raise RuntimeError('통합 ZIP 안의 Code_v3.zip을 유일하게 찾지 못했습니다.')
        repo_bytes=outer.read(matches[0])
    archive=zipfile.ZipFile(io.BytesIO(repo_bytes))
else:
    archive=zipfile.ZipFile(REPO_ZIP)

# 이전 실패로 생긴 불완전한 폴더를 제거한 후 깨끗하게 재추출.
if WORK.exists():
    shutil.rmtree(WORK)
with archive as z:
    entries=z.namelist()
    expected='CSE304_Xray_Colab_Repo/'
    for info in z.infolist():
        name=info.filename
        rel=Path(name)
        if not name.startswith(expected) or rel.is_absolute() or '..' in rel.parts:
            raise RuntimeError(f'예상하지 못한 ZIP 내부 경로: {name}')
        if info.is_dir():
            continue
        target=(ROOT/rel).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise RuntimeError(f'잘못된 ZIP 경로: {name}')
        target.parent.mkdir(parents=True,exist_ok=True)
        with z.open(info) as source, open(target,'wb') as dest:
            shutil.copyfileobj(source,dest)

for file in ['requirements.txt','pyproject.toml','run.py']:
    if not (WORK/file).is_file():
        raise FileNotFoundError(f'압축 해제 결과 누락: {WORK/file}')
if not INPUT.is_dir():
    raise FileNotFoundError(f'입력 데이터 폴더 없음: {INPUT}')
zips=list(INPUT.glob('*.zip'))
if len(zips)!=9:
    raise RuntimeError(f'원본 ZIP 8개 + 11/10 검수 ZIP 1개, 총 9개 필요. 현재 {len(zips)}개: {[p.name for p in zips]}')
review=[p for p in zips if 'xray-labels' in p.name.lower()]
if len(review)!=1 or '2026-11-10' not in review[0].name:
    raise RuntimeError('검수 파일은 xray-labels-2026-11-10.zip 하나만 넣으세요.')
OUTPUT.mkdir(parents=True,exist_ok=True)
print('레포지토리 ZIP:',REPO_ZIP)
print('압축 해제 폴더:',WORK)
print('requirements:',(WORK/'requirements.txt').exists())
print('입력 ZIP:',len(zips),'개')


In [ ]:
import os, sys, subprocess
from pathlib import Path
assert (WORK/'requirements.txt').is_file(), f'코드 설치 경로 확인 실패: {WORK}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK/'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(WORK)], check=True)
print('설치 완료:', WORK)


## 실행 설정

처음에는 `FAST_TEST=True`, `USE_DEEP=False`로 입력·라벨·박스 제거를 검증하고, 전체 실행 때 `FAST_TEST=False`, `USE_DEEP=True`, `USE_YOLO_ROI=True`로 설정하세요. Google Colab 메뉴에서 GPU 런타임을 켜는 것을 권장합니다.


In [ ]:
import os, sys, subprocess
FAST_TEST = True
USE_DEEP = False
USE_YOLO_ROI = False
FAST_TEST_LIMIT = 72
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
cmd=[sys.executable, str(WORK/'run.py'), '--input',str(INPUT),'--output',str(OUTPUT),'--work','/content/xray_v3_work']
if FAST_TEST: cmd+=['--max-images',str(FAST_TEST_LIMIT)]
if not USE_DEEP: cmd+=['--no-deep']
if USE_YOLO_ROI: cmd+=['--with-yolo-roi']
print('Command:', ' '.join(cmd))
subprocess.run(cmd,check=True,cwd=str(WORK),env=os.environ.copy())


## 결과 확인

최초 QC는 `quality_control/overlay`의 원본·마스크·보간 이미지, `audit/label_audit.json`, `audit/groups/group_label_conflicts.csv`를 직접 보면서 진행하세요. FAST_TEST에서는 검수 행 전체와 일부 원본만 불러오므로 미매칭 감사 항목이 많을 수 있습니다. 전체 실행에서만 전체 매칭이 기대됩니다.


In [ ]:
from pathlib import Path
import pandas as pd,json
print(json.loads((OUTPUT/'audit/label_audit.json').read_text()))
for rel in ['audit/groups/group_label_conflicts.csv',
            'analysis/e4/e4_group_bootstrap_statistics.csv',
            'analysis/ablations/all_reviewed/feature_family_ablation.csv',
            'analysis/ablations/all_reviewed/annotation_coverage_by_label.csv']:
    p=OUTPUT/rel
    print('\n',rel,'exists:',p.exists())
    if p.exists():display(pd.read_csv(p).head(12))
from IPython.display import display,Image
pictures=sorted((OUTPUT/'quality_control/overlay').glob('overlay_*.png'))
if pictures:display(Image(filename=str(pictures[0]),width=1100))


## 해석 한계

- 본 프로젝트는 NG 영상의 **표시 적절성** 연구입니다.
- F 중에는 다양한 예외가 섞여 있고, 지원 박스 0개라는 이유만으로 프레임 이상이라고 확정하지 않습니다.
- 인페인팅은 원본 X-ray를 복구하지 않습니다.
- 후보별 covered/missed의 검증된 정답이 없으므로 후보 수준 미검출률을 보고하지 않습니다.
- 전체 GPU 학습 결과를 생성하기 전까지 실제 AP와 p-value를 발표 자료에 기입하지 마세요.
